# Concrete Compressive Strength Prediction

**Dataset:** UCI Concrete — 1,030 laboratory mixes with their ingredients (cement, slag, fly ash, water, superplasticizer, aggregates), curing age in days, and the measured compressive strength in MPa. **Task:** predict strength from the recipe — the question a civil engineer asks before pouring.

In [1]:
import io, urllib.request, zipfile
import pandas as pd

def get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=90) as r: return r.read()

zf = zipfile.ZipFile(io.BytesIO(get('https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip')))
name = [n for n in zf.namelist() if n.endswith('.xls')][0]
d = pd.read_excel(io.BytesIO(zf.read(name)))
d.columns = ['cement', 'slag', 'flyash', 'water', 'superplasticizer', 'coarse_agg', 'fine_agg', 'age', 'strength']
print(d.shape)
d.head()

(1030, 9)


,cement,slag,flyash,water,superplasticizer,coarse_agg,fine_agg,age,strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X, y = d.drop(columns=['strength']), d['strength']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lin = LinearRegression().fit(Xtr, ytr)
pred = rf.predict(Xte)
print('Random Forest R2:', round(r2_score(yte, pred), 4), '| MAE (MPa):', round(mean_absolute_error(yte, pred), 2))
print('Linear Regression R2:', round(r2_score(yte, lin.predict(Xte)), 4))
imp = sorted(zip(X.columns, rf.feature_importances_), key=lambda t: -t[1])
print('Feature importance:', [(k, round(v, 3)) for k, v in imp])

Random Forest R2: 0.8906 | MAE (MPa): 3.72
Linear Regression R2: 0.6249
Feature importance: [('age', np.float64(0.33)), ('cement', np.float64(0.325)), ('water', np.float64(0.113)), ('superplasticizer', np.float64(0.073)), ('slag', np.float64(0.071)), ('fine_agg', np.float64(0.037)), ('coarse_agg', np.float64(0.028)), ('flyash', np.float64(0.022))]


## What I learned
- Age (curing time) and cement content dominate — concrete keeps gaining strength as it cures, and the forest captures that non-linear curve while linear regression cannot.
- Water works against strength (the classic water–cement ratio), visible in the importance ranking.
- An MAE of a few MPa is useful for recipe screening, not for certifying a structure — lab mixes vary less than site concrete.